# Stage-2 retrain — `traffic-yolo-augmented` (Phase 8)

Trains YOLO11n on the project's stage-2 dataset (`lynkeus03/vehicle-detection-by9xs` v3, 9,211 images, 6 classes) with augmentation deliberately tuned for small/distant-vehicle detection, and logs the run to MLflow as experiment `traffic-yolo-augmented`. Uses the exact same `training/train.py` and `training/config_stage2_augmented.yaml` that live in the repo — this notebook is just the runner, no training logic is duplicated here.

**Before running:**
1. In Colab: **Runtime -> Change runtime type -> T4 GPU** (or better).
2. Have your Roboflow API key ready (from `app.roboflow.com/settings/api`). It is **never** typed into a cell or printed anywhere in this notebook — see the "Roboflow API key" section below for how it's handled.
3. Have `traffic_classifier_code.zip` ready to upload (built from the project root — see repo README if you need to regenerate it: it's just `src/`, `training/`, `evaluation/`, `requirements.txt`, `.env.example`, zipped).

Expected runtime: with a T4 GPU, ~9,200 images / batch 16 / 100 epochs is a multi-hour run, not a quick one — see the "Save to Google Drive" toggle below before starting, so a Colab disconnect doesn't lose progress.

In [ ]:
# Confirm a GPU is actually attached. If this errors or shows no GPU,
# stop and fix Runtime -> Change runtime type -> GPU before continuing —
# training this on CPU would take days, not hours.
!nvidia-smi

## 1. Get the project code onto Colab

No GitHub remote is set up for this project yet, so the low-friction path is: upload the small code-only zip built from the repo (`notebooks/traffic_classifier_code.zip`, ~45KB — just `src/`, `training/`, `evaluation/`, `requirements.txt`, `.env.example`; no data, no weights, no secrets). If you've since pushed the repo to GitHub, swap the upload cell below for a `git clone` instead — both land in the same place.

**Save to Google Drive?** A 100-epoch GPU run can take a few hours; a Colab disconnect mid-run without this loses everything. Recommended: leave `SAVE_TO_DRIVE = True` (default) so the whole working directory — code, dataset, checkpoints, MLflow db — lives on your Drive and survives a disconnect. Set to `False` only for a quick smoke test you don't mind losing.

In [ ]:
SAVE_TO_DRIVE = True  # recommended for the real 100-epoch run; False is fine for a quick smoke test

import os

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/real-time-traffic-classifier-colab'
else:
    PROJECT_DIR = '/content/real-time-traffic-classifier-colab'

os.makedirs(PROJECT_DIR, exist_ok=True)
%cd $PROJECT_DIR
print('Working directory:', PROJECT_DIR)

In [ ]:
# --- Option A (default): upload the code zip ---
# Run this cell, then use the file picker to select traffic_classifier_code.zip
# from your local machine (notebooks/traffic_classifier_code.zip in the repo).
# Safe to re-run — it overwrites in place, so re-uploading after a local code
# change (e.g. re-tuning an augmentation value) just works.
import zipfile
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('.')
print('Extracted:', zip_name)

In [ ]:
# --- Option B: clone from GitHub instead (use this cell INSTEAD of the upload
# cell above, once the repo has a GitHub remote) ---
# !git clone https://github.com/<your-username>/real-time-traffic-classifier.git .
print('Skipped — using the zip-upload path above. Uncomment this cell to use git clone instead.')

In [ ]:
# Install project dependencies. Colab already ships a CUDA-enabled torch —
# deliberately NOT reinstalling torch/torchvision here (unlike the project's
# local CPU-only setup) so Colab's own matching CUDA build stays intact.
!pip install -q ultralytics mlflow roboflow python-dotenv pyyaml

## 2. Roboflow API key

**Never typed into a cell, never printed, never committed.** Preferred path: add it once as a Colab secret (left sidebar -> key icon -> "Add new secret", name it `ROBOFLOW_API_KEY`, toggle notebook access on) — then the cell below reads it silently. If you haven't set up the secret, it falls back to a masked `getpass` prompt instead. Either way, the key is written only to a local `.env` file inside `PROJECT_DIR` (which `training/download_dataset.py` already reads via `load_dotenv()` — same mechanism as the local project setup), never displayed in this notebook's output.

In [ ]:
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    pass

if not api_key:
    import getpass
    api_key = getpass.getpass('Roboflow API key (input hidden): ')

with open('.env', 'w') as f:
    f.write(f'ROBOFLOW_API_KEY={api_key}\n')

api_key = None  # drop the in-memory copy once it's written to .env
print('.env written.', 'Length check only, never the value:', 'OK' if os.path.getsize('.env') > len('ROBOFLOW_API_KEY=\n') else 'EMPTY — key was blank')

## 3. Download + prepare the stage-2 dataset

Same two-step pipeline as local training (`training/download_dataset.py` then `training/prepare_dataset.py`) — nothing Colab-specific here, just run on this machine instead. `prepare_dataset.py` writes `data/dataset_stage2.yaml` with an absolute path resolved fresh in *this* environment, so it correctly points at Colab's filesystem, not the original Windows dev machine's.

In [ ]:
!python -m training.download_dataset --workspace lynkeus03 --project vehicle-detection-by9xs --version 3

In [ ]:
!python -m training.prepare_dataset \
  --source data/raw/vehicle-detection-by9xs \
  --output-dir data/processed_stage2 \
  --dataset-yaml data/dataset_stage2.yaml

In [ ]:
# Sanity check before committing to a long run — same principle as every
# other phase of this project: verify before proceeding, not after.
!cat data/dataset_stage2.yaml
!echo '---'
!echo "train images: $(ls data/processed_stage2/images/train | wc -l)"
!echo "val images:   $(ls data/processed_stage2/images/val | wc -l)"
!echo "test images:  $(ls data/processed_stage2/images/test | wc -l)"

## 4. Smoke test first (2 epochs)

Before committing GPU-hours to the full 100-epoch run, run 2 epochs end-to-end — catches config/path/dependency problems in minutes instead of hours. This mirrors the same local smoke test already run for the augmentation wiring itself (`--name smoke_test_aug_wiring`, see README) — same idea, now against the real stage-2 data on the real target hardware.

In [ ]:
!python -m training.train --config training/config_stage2_augmented.yaml \
  --epochs 2 --name stage2_smoke_test --mlflow-experiment traffic-yolo-augmented-smoke

Check the cell output above: training should complete both epochs, print a `Test metrics:` line, and end with `Training complete`. If it errored, fix that before running the real thing below — a failure at epoch 98 of 100 is a much more expensive way to find the same bug.

## 5. Full training run (100 epochs)

This is the real run — everything from `training/config_stage2_augmented.yaml` (tuned augmentation: `scale=0.9`, `translate=0.2`, `hsv_v=0.5`, `copy_paste=0.3`; see the repo README's "Stage-2 retrain" section for why each was chosen), logged to MLflow as `traffic-yolo-augmented`. Colab can idle-disconnect after a period of inactivity — if `SAVE_TO_DRIVE = True` above, checkpoints and the MLflow db live on Drive and survive that; if it does disconnect, just re-run this cell (Ultralytics resumes from `last.pt` under the same run name automatically when `exist_ok=True`, which `train.py` already sets).

In [ ]:
!python -m training.train --config training/config_stage2_augmented.yaml

## 6. Review results

Prints the same params/metrics logged to MLflow (precision/recall/mAP50/mAP50-95/FPS/latency), read straight from the run — no need to stand up `mlflow ui` on Colab just to see the numbers.

In [ ]:
import mlflow

mlflow.set_tracking_uri('sqlite:///mlflow.db')
client = mlflow.MlflowClient()
exp = client.get_experiment_by_name('traffic-yolo-augmented')
runs = client.search_runs(exp.experiment_id, order_by=['start_time DESC'], max_results=1)
run = runs[0]

print('Run:', run.info.run_id, '-', run.info.status)
print('\nParams:')
for k, v in sorted(run.data.params.items()):
    print(f'  {k}: {v}')
print('\nMetrics:')
for k, v in sorted(run.data.metrics.items()):
    print(f'  {k}: {v:.4f}')

## 7. Get the weights back

Zips `best.pt` plus the MLflow tracking db (so the run's params/metrics travel with the weights, not just the weights alone), then offers it as a browser download. If `SAVE_TO_DRIVE = True`, everything is already sitting on your Drive too — this download is just for convenience.

In [ ]:
import shutil
from pathlib import Path

best_pt = Path('outputs/training_runs/stage2_augmented/weights/best.pt')
assert best_pt.exists(), f'{best_pt} not found — did the full training run (section 5) complete?'

bundle_dir = Path('stage2_augmented_bundle')
bundle_dir.mkdir(exist_ok=True)
shutil.copy2(best_pt, bundle_dir / 'best.pt')
shutil.copy2('mlflow.db', bundle_dir / 'mlflow.db')

archive = shutil.make_archive('stage2_augmented_bundle', 'zip', bundle_dir)
print('Bundle ready:', archive)

from google.colab import files
files.download(archive)

## 8. Next steps (back on the local project)

1. Unzip the downloaded bundle; drop `best.pt` into the local project's `models/` directory (e.g. `models/stage2_augmented.pt`).
2. `evaluation/compare_models.py` can then produce a baseline-vs-stage2 comparison the same way the existing stage-1 Results table was built — note its `evaluation/coco_overlap.py` class-name mapping currently only covers stage-1's class names (Ambulance/Bus/Car/Motorcycle/Truck) and will need `bus/car/motorbike/truck` added for stage-2's names (`microbus`/`pickup-van` have no COCO equivalent, same situation as `Ambulance` did).
3. Point `dashboard/app.py` / `src/inference.py --model` at the new weights to see it on real footage (`golden_gate_bridge_night.webm` is the natural first test, given this whole retrain traces back to that clip's small/distant-vehicle finding).
4. `mlflow.db` from the bundle can be merged/compared against the local one if you want the run visible in a local `mlflow ui` too — otherwise the printed metrics above are already the full record.